# Study Assistant Quiz Generator (PDF -> Summary -> MCQ)

This notebook loads `prompt_engineering.pdf`, summarizes the content into bullet points, then generates MCQs with answers using LangChain and Google Gemini (gemini-2.5-flash).

API key will be requested securely with `getpass`.

In [ ]:
# Install dependencies (run once)
# ! pip install -q langchain langchain-google-genai pypdf2


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [17]:
# ! pip install langchain-community
# ! pip install pypdf

In [ ]:
# Read the PDF and extract text
from langchain_community.document_loaders import PyPDFLoader

# Load PDF content
pdf_path = 'prompt_engineering.pdf'
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# combine all pages text into one body
pdf_text = '\n\n'.join([doc.page_content for doc in documents])

print(f'Loaded {len(documents)} pages, total text length: {len(pdf_text)} chars')

Loaded 7 pages, total text length: 4312 chars


In [20]:
print(pdf_text)

What isPrompt Engineering?
Promptengineeringisapracticewithinnaturallanguageprocessing(NLP)inartificial
intelligence,wheretextisusedtodescribethetasktheAIshouldperform.Guidedby
thisinput,theAIgeneratesanoutput,whichcouldtakevariousforms.Thegoalisto
usehuman-understandabletexttointeractconversationallywithmodels,allowingfor
flexibilityinthemodel’sperformanceduetothetaskdescriptionembeddedinthe
prompt.
What arePrompts?
PromptsaredetaileddescriptionsofthedesiredoutputfromanAImodel.They
representtheinteractionbetweentheuserandthemodelandhelpdefinewhattheAIis
expectedtodo.Theeffectivenessofpromptengineeringlargelydependsonhowwell
thepromptisdesignedtoguidethemodel.
Examplesof Prompt Engineering
Promptsinlargelanguagemodels(LLMs)likeChatGPTorGPT-3canrangefrom
simpletextqueriestocomplexinstructions.Thekeytoeffectivepromptingisproviding
sufficientdetail.Examplesofpromptsforvarioustasksinclude:
Text Prompts(ChatGPT, GPT):

● "What’sthedifferencebetweengenerativeAIandtraditionalAI?"
● "Provide10

In [21]:
len(pdf_text)

4312

In [3]:
from getpass import getpass
import os

Google_KEY = getpass('Enter Google API Key: ')
os.environ['GOOGLE_API_KEY'] = Google_KEY

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM initialization with Gemini
llm_model = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers.string import StrOutputParser

# Chain 1: summarization into bullet points
summary_prompt = '''You are an expert tutor. Read the following text and produce a concise summary in 8-12 bullet points. Keep each bullet short and focused. Use numbered bullets.\n\nText:\n{text}\n\nSummary in points:\n'''

summary_prompt_template = PromptTemplate(
    input_variables=['text'],
    template=summary_prompt
)

summary_chain = (summary_prompt_template | llm_model | StrOutputParser() )

summary = summary_chain.invoke({"text": pdf_text})

print(summary)


Here is a concise summary of the text in 9 bullet points:

1.  Prompt engineering is an NLP practice where text describes tasks for AI models to perform.
2.  Prompts are detailed descriptions that define the desired output from an AI model.
3.  The effectiveness of prompt engineering depends on how well prompts are designed to guide the model.
4.  Examples of prompts include text queries (ChatGPT), code instructions (Codex), and image descriptions (Stable Diffusion).
5.  To improve prompt quality, consider role-playing, clarity, specificity, and consistency.
6.  Key elements of a prompt include instruction, context, input data, and an output indicator.
7.  Standard prompt patterns include user-model interaction, few-shot prompting, and question-and-answer formats.
8.  Advanced prompting techniques include Zero-Shot, Few-Shot (in-context learning), and Chain-of-Thought.
9.  Avoid information overload, open-ended questions, and poor use of constraints when crafting prompts.


In [26]:
# Chain 2: MCQ generation from summary points
mcq_prompt = PromptTemplate(
    input_variables=['summary'],
    template='''You are a quiz maker. Use the following summary points and generate 5 multiple-choice questions.\nFor each question, provide 4 options (A-D) and mark the correct answer clearly.\nOutput format:\nQuestion N:\nA) ...\nB) ...\nC) ...\nD) ...\nAnswer: <letter> | <text>\n\nSummary points:\n{summary}\n\nMCQs:\n'''
)

mcq_chain = ( mcq_prompt | llm_model | StrOutputParser() )

mcq = mcq_chain.invoke({"summary": summary})

print(mcq)

# # Combine into a sequential chain
# seq_chain = SequentialChain(
#     chains=[summary_chain, mcq_chain],
#     input_variables=['text'],
#     output_variables=['summary', 'mcq'],
#     verbose=True
# )

# result = seq_chain.run(text=pdf_text)
# print('--- Result ---')
# print(result)

**Question 1:**
What is the primary purpose of prompt engineering in the context of AI models?
A) To develop new AI models from scratch.
B) To provide detailed text descriptions that define desired outputs and tasks for AI models.
C) To analyze the internal algorithms of AI models.
D) To solely focus on image generation for AI models.
Answer: B | To provide detailed text descriptions that define desired outputs and tasks for AI models.

**Question 2:**
Which of the following sets accurately represents the key elements of a prompt?
A) Model architecture, training data, validation metrics, and deployment strategy.
B) Instruction, context, input data, and an output indicator.
C) User interface, database schema, API endpoints, and server logs.
D) Programming language, compiler, debugger, and version control.
Answer: B | Instruction, context, input data, and an output indicator.

**Question 3:**
Which of the following are considered advanced prompting techniques?
A) User-model interaction a

In [27]:
from operator import itemgetter

combined_chain = (
    {
        "input": itemgetter('text'),
        "summary": summary_chain,
    }
    |
    mcq_chain
)

In [28]:
response = combined_chain.invoke({"text": pdf_text})
print(response)

Question 1:
What is the primary definition of Prompt Engineering according to the summary?
A) A method for designing new AI model architectures.
B) An NLP practice where text describes tasks for an AI to perform.
C) A technique for optimizing AI hardware performance.
D) A process for collecting and labeling large datasets for AI training.
Answer: B | An NLP practice where text describes tasks for an AI to perform.

Question 2:
Which of the following is NOT listed as a type of output that prompts can be used to generate?
A) Text
B) Code
C) Images
D) Hardware specifications
Answer: D | Hardware specifications

Question 3:
According to the summary, which of these is a key element that should be included in a prompt?
A) User interface design
B) Output indicator
C) AI model version
D) Training data source
Answer: B | Output indicator

Question 4:
Which of the following is identified as an advanced prompting technique?
A) User-Model Interaction
B) Few-Shot Prompting
C) Standard Prompting
D) 

In [ ]:
# If result is dict-like, print structured outputs
if isinstance(result, dict):
    print('\nSummary:\n', result.get('summary', ''))
    print('\nMCQs:\n', result.get('mcq', ''))